# 99 - Robustness Checks

Sensitivity of main TWFE and C-S results to: control group, outcome definition, anticipation effects, and border-state spillovers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"
fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
never    = leg[leg['retail_sales_year'].isna()]['state'].tolist()
in_window = leg[leg['retail_sales_year'].between(2010,2022)]['state'].tolist()

def twfe_att(data, outcome, treat_col='post'):
    idx = data.set_index(['state','year'])
    fe  = PanelOLS(idx[outcome], idx[[treat_col]],
                   entity_effects=True, time_effects=True).fit(
                   cov_type='clustered', cluster_entity=True)
    b, ci = fe.params[treat_col], fe.conf_int().loc[treat_col]
    return b, ci['lower'], ci['upper']

# Baseline
base_data = fars_reg[fars_reg['state'].isin(in_window + never)].copy()
base_data['post'] = (
    base_data['retail_sales_year'].notna() &
    (base_data['year'] >= base_data['retail_sales_year'])
).astype(float)

b_base, lo, hi = twfe_att(base_data, primary)
print(f"Baseline TWFE ATT: {b_base:.4f}  [{lo:.4f}, {hi:.4f}]")

## Check 1 - Exclude early adopters (CO, WA)

In [ ]:
early = leg[leg['retail_sales_year'] <= 2015]['state'].tolist()
sub = base_data[~base_data['state'].isin(early)].copy()
b, lo, hi = twfe_att(sub, primary)
print(f"Excl. early adopters: {b:.4f}  [{lo:.4f}, {hi:.4f}]")
print(f"Baseline:             {b_base:.4f}")

## Check 2 - Anticipation: shift treatment date 1 year earlier

In [ ]:
anticipation_data = fars_reg.copy()
anticipation_data['retail_sales_year_adj'] = anticipation_data['retail_sales_year'] - 1
anticipation_data['post_adj'] = (
    anticipation_data['retail_sales_year_adj'].notna() &
    (anticipation_data['year'] >= anticipation_data['retail_sales_year_adj'])
).astype(float)
sub_a = anticipation_data[anticipation_data['state'].isin(in_window + never)].copy()
b_a, lo_a, hi_a = twfe_att(sub_a, primary, 'post_adj')
print(f"Anticipation (1yr early): {b_a:.4f}  [{lo_a:.4f}, {hi_a:.4f}]")
print(f"Baseline:                 {b_base:.4f}")
print()
print("If results change substantially, there may be pre-enforcement behavioral responses.")

## Check 3 - Restrict to contiguous US (exclude AK, HI)

In [ ]:
sub_cont = base_data[~base_data['state'].isin(['Alaska','Hawaii'])].copy()
b_c, lo_c, hi_c = twfe_att(sub_cont, primary)
print(f"Contiguous US only: {b_c:.4f}  [{lo_c:.4f}, {hi_c:.4f}]")
print(f"Baseline:           {b_base:.4f}")

## Check 4 - Alternative outcome: alcohol-impaired fatalities

In [ ]:
for outcome in ['alcohol_fatalities_per_100k','drug_fatalities_per_100k']:
    if outcome not in base_data.columns:
        print(f"  {outcome} not in panel — add to build step")
        continue
    b_o, lo_o, hi_o = twfe_att(base_data, outcome)
    print(f"{outcome:<40}: ATT = {b_o:+.4f}  [{lo_o:+.4f}, {hi_o:+.4f}]")

## Summary table

In [ ]:
checks = [
    ("Baseline TWFE",                b_base, lo, hi),
    ("Excl. early adopters",         b, lo, hi),
    ("Anticipation (1yr early)",     b_a, lo_a, hi_a),
    ("Contiguous US only",           b_c, lo_c, hi_c),
]
print(f"{'Specification':<35}  {'ATT':>8}  {'CI'}")
print("-"*60)
for name, b, lo_, hi_ in checks:
    print(f"{name:<35}  {b:>8.4f}  [{lo_:+.4f}, {hi_:+.4f}]")